# distance

## prepare

In [84]:
import os
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from scipy.spatial.distance import cosine
from scipy.spatial import distance

In [85]:
import warnings

# RuntimeWarning 무시
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [86]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [87]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(study1_dir + model_path, binary=True)

In [88]:
# 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', encoding='ISO-8859-1')

In [89]:
seed_words = ['key', 'money', 'friend']
target_words = ['money', 'friend']

n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(tbl_data) # 210
n_dim_of_vector = 300

## 벡터 구하기

In [90]:
for seed_word in seed_words: # key, money, friend
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # key1~30, money1~30, friend1~30

    for column in word_columns:
        # vector field 생성
        tbl_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # 피험자 한 명의 응답 단어들 벡터 처리
        for i_subject in range(n_subject):
            try:
                response_word = tbl_data.iloc[i_subject][column]
                if pd.isna(response_word) or len(response_word.strip()) == 0:# NaN, 값이 빈 칸 & 응답안해서 '', ' '로 저장된 경우 걸러내기
                    tbl_data[column + '_vec'][i_subject] = None
                    continue

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]
                    if len(response_word) == 0: # 앞에서 걸러져서 결과가 없으면, 넘어가기
                        tbl_data[column + '_vec'][i_subject] = None
                        continue

                    vec_word2vec = np.zeros((n_dim_of_vector, 0))  # 300차원의 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        # reshape: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기
                    # 각 열(단어 벡터)에 대한 평균 계산
                    average_vector = np.mean(vec_word2vec, axis=1)
                    tbl_data[column + '_vec'][i_subject] = average_vector
            except:
                pass

/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/us

## Word Mover’s Distance(WMD)

NLP연구쪽에서 문서 유사도 구하는 방법으로 잘 쓰인다고 함. (논문 연구 결과 다른 방법론보다 성능이 좋다고 함)\
기본적으로 word2vec을 베이스로 하고, word2vec을 사용해서 word간의 euclide distance를 구함!

- 피험자 1명, seed 단어 1개

피험자가 대답한 30개의 연속적인 단어 그룹 - target word(money, friend)의 유사도를 기반으로 wmdistance를 잰다.

--> 그럼 2개의 distance값이 나옴. 그 중 가장 값이 작은 것이 거리가 가깝다. 즉, target word와 유사하다.

wmd 설명 참조
- https://sy-programmingstudy.tistory.com/14
- https://www.youtube.com/watch?v=zFnrq5SmBdg

In [91]:
# wmd_seed_target 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        column_name = f'wmd_{seed_word}_{target_word}'
        tbl_data[column_name] = np.nan
    column_name = f'wmd_{target_word}'
    tbl_data[column_name] = np.nan

tbl_data.columns

Index(['subject', 'key1', 'key2', 'key3', 'key4', 'key5', 'key6', 'key7',
       'key8', 'key9',
       ...
       'friend29_vec', 'friend30_vec', 'wmd_key_money', 'wmd_money_money',
       'wmd_friend_money', 'wmd_money', 'wmd_key_friend', 'wmd_money_friend',
       'wmd_friend_friend', 'wmd_friend'],
      dtype='object', length=189)

In [92]:
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
# word_columns

In [93]:
for i_subject in range(n_subject):
    # print('subject: ', i_subject)

    for target_word in target_words: # money, friend
        for seed_word in seed_words: # key, money, friend
            # 각 피험자의 응답 문서를 30개씩 단어 리스트로 반환
            seed_response_words = [tbl_data.iloc[i_subject][column] for column in word_columns if column.startswith(seed_word)]
            # nan값 걸러내기
            seed_response_words = [word for word in seed_response_words if not isinstance(word, float) or not np.isnan(word)] 

            if len(seed_response_words) > 0: # 애초에 값이 0인 피험자 데이터들은 건너뛰도록 함
                try:
                    # 응답 30개 단어 - target 단어의 WMD 계산
                    wmdistance_value = word2vec_model.wmdistance([target_word], seed_response_words)

                    # 계산한 WMD를 해당 테이블 위치에 저장
                    tbl_data.at[i_subject, f'wmd_{seed_word}_{target_word}'] = wmdistance_value
                    # print(f'wmd_{seed_word}_{target_word}: {wmdistance_value}')
                except Exception as e:
                    print(f"An error occurred for subject {i_subject}: {str(e)}")
                    continue

        # 각 피험자의 응답 단어 90개 모두를 리스트로 반환
        total_response_words = [tbl_data.iloc[i_subject][column] for column in word_columns]
        # 응답 90개 단어 - target 단어의 WMD 계산
        total_wmdistance_value = word2vec_model.wmdistance([target_word], total_response_words)

        # 계산한 WMD를 해당 테이블 위치에 저장
        tbl_data.at[i_subject, f'wmd_{target_word}'] = total_wmdistance_value
        # print(f'wmd_{target_word}: {total_wmdistance_value}')


In [94]:
tbl_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,friend29_vec,friend30_vec,wmd_key_money,wmd_money_money,wmd_friend_money,wmd_money,wmd_key_friend,wmd_money_friend,wmd_friend_friend,wmd_friend
0,1,card,bank,money,green,yellow,blue,eye,nose,smell,...,"[-0.08740234375, -0.1337890625, -0.044921875, ...","[-0.21875, -0.1533203125, -0.23828125, 0.33789...",1.286583,1.333494,1.365774,1.328199,1.331864,1.332841,1.354942,1.339713
1,2,lock,door,big,lion,roar,scary,ghost,dark,halloween,...,"[0.111328125, 0.10595703125, -0.07373046875, 0...","[0.05126953125, -0.0223388671875, -0.172851562...",1.346666,1.317465,1.350067,1.338066,1.346911,1.340560,1.323844,1.337105
2,3,door,window,curtain,draft,cold,snow,scarf,wooly hat,winter,...,"[0.138671875, 0.2041015625, 0.0289306640625, 0...","[0.1787109375, 0.326171875, -0.11865234375, 0....",1.332693,1.265375,1.348246,1.316453,1.287123,1.327414,1.332750,1.316757


# distances between target word

## average distance between all response words and target word

90개의 응답 단어들 각각과 타겟 단어와의 거리의 평균을 구한다.

In [95]:
distances = []

for i_subject in range(n_subject):
    try:
        response_words = tbl_data.iloc[i_subject, 91:181].tolist()
        response_words_cluster = np.array([word_vec for word_vec in response_words if not (np.any(pd.isna(word_vec)))])
        
        for target_word in target_words: # money, friend
            target_word_vec = np.array(word2vec_model[target_word])

            # # 각 response_words 벡터와 target_word_vec 간의 거리 계산 및 리스트에 추가 -> 평균 계산
            distances = [cosine(word_vec, target_word_vec) for word_vec in response_words_cluster]
            average_distance = np.mean(distances)
            tbl_data.at[i_subject, f'distance_average_{target_word}'] = average_distance
        
    except Exception as e:
        print(f'subject {i_subject+1}의 응답단어 클러스터와 {target_word}간의 distance 계산에 실패했습니다. e: {e}')


/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/2075321321.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tbl_data.at[i_subject, f'distance_average_{target_word}'] = average_distance
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/2075321321.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tbl_data.at[i_subject, f'distance_average_{target_word}'] = average_distance


In [96]:
tbl_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,wmd_key_money,wmd_money_money,wmd_friend_money,wmd_money,wmd_key_friend,wmd_money_friend,wmd_friend_friend,wmd_friend,distance_average_money,distance_average_friend
0,1,card,bank,money,green,yellow,blue,eye,nose,smell,...,1.286583,1.333494,1.365774,1.328199,1.331864,1.332841,1.354942,1.339713,0.893631,0.899278
1,2,lock,door,big,lion,roar,scary,ghost,dark,halloween,...,1.346666,1.317465,1.350067,1.338066,1.346911,1.340560,1.323844,1.337105,0.897881,0.896145
2,3,door,window,curtain,draft,cold,snow,scarf,wooly hat,winter,...,1.332693,1.265375,1.348246,1.316453,1.287123,1.327414,1.332750,1.316757,0.875426,0.862669


## distance between cluster and target word

90개의 응답단어들을 하나의 클러스터로 보고, 해당 클러스터와 타겟 단어간의 거리를 구한다.

참고: https://zephyrus1111.tistory.com/180

In [97]:
## vector가 모두 300차원인지 확인하는 코드
# import numpy as np

# # 데이터 프레임에서 벡터가 저장된 열을 선택 (예: 161번부터 320번 열)
# vector_column = tbl_data.columns[161:321]

# for i_subject in range(n_subject):
#     response_words_cluster = tbl_data.iloc[i_subject][vector_column].tolist()

#     for i, vector_str in enumerate(response_words_cluster):
#         if vector_str is not None:
#             vector = np.fromstring(vector_str, sep=' ')
#             if vector_str.shape[0] != 300:
#                 print(f"Vector in subject {i_subject}, row {i} has {vector.shape[0]} dimensions.")


In [98]:
# distance_centroid_target, distance_medoid_target 컬럼 미리 생성(빈 값)
for target_word in target_words:
    column_name = f'distance_centroid_{target_word}'
    tbl_data[column_name] = np.nan

for target_word in target_words:
    column_name = f'distance_medoid_{target_word}'
    tbl_data[column_name] = np.nan

tbl_data.columns

/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/4031724281.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tbl_data[column_name] = np.nan
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/4031724281.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tbl_data[column_name] = np.nan
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_12667/4031724281.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p

Index(['subject', 'key1', 'key2', 'key3', 'key4', 'key5', 'key6', 'key7',
       'key8', 'key9',
       ...
       'wmd_key_friend', 'wmd_money_friend', 'wmd_friend_friend', 'wmd_friend',
       'distance_average_money', 'distance_average_friend',
       'distance_centroid_money', 'distance_centroid_friend',
       'distance_medoid_money', 'distance_medoid_friend'],
      dtype='object', length=195)

In [99]:
for i_subject in range(n_subject):
    # print('subject: ', i_subject)
    try:
        response_words = tbl_data.iloc[i_subject, 91:181].tolist()
        # NaN 값을 제외하고 NumPy 배열로 변환
        response_words_cluster = np.array([word_vec for word_vec in response_words if not (np.any(pd.isna(word_vec)))])

        # 클러스터의 중심(centroid: 데이터 포인트의 평균) 계산
        centroid = np.nanmean(response_words_cluster, axis=0)

        # 클러스터의 중앙값(medoid: 데이터 세트 내의 가장 중앙에 있는 데이터 포인트) 계산
        cosine_distances_matrix = cosine_distances(response_words_cluster)
        medoid = response_words_cluster[np.argmin(np.sum(cosine_distances_matrix, axis=0))]

        for target_word in target_words: # money, friend, relationships, family
            target_word_vec = np.array(word2vec_model[target_word])

            cosine_distance_with_centroid = 1 - cosine_similarity(centroid.reshape(1, -1), target_word_vec.reshape(1, -1))
            cosine_distance_with_medoid = 1 - cosine_similarity(medoid.reshape(1, -1), target_word_vec.reshape(1, -1))

            tbl_data.at[i_subject, f'distance_centroid_{target_word}'] = cosine_distance_with_centroid[0][0]
            tbl_data.at[i_subject, f'distance_medoid_{target_word}'] = cosine_distance_with_medoid[0][0]

    except Exception as e:
        print(f'subject {i_subject+1}의 응답단어 클러스터와 {target_word}간의 distance 계산에 실패했습니다. e: {e}')
        continue


In [100]:
tbl_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,wmd_key_friend,wmd_money_friend,wmd_friend_friend,wmd_friend,distance_average_money,distance_average_friend,distance_centroid_money,distance_centroid_friend,distance_medoid_money,distance_medoid_friend
0,1,card,bank,money,green,yellow,blue,eye,nose,smell,...,1.331864,1.332841,1.354942,1.339713,0.893631,0.899278,0.696055,0.701433,0.921414,0.919605
1,2,lock,door,big,lion,roar,scary,ghost,dark,halloween,...,1.346911,1.340560,1.323844,1.337105,0.897881,0.896145,0.723237,0.712520,1.008306,0.888915
2,3,door,window,curtain,draft,cold,snow,scarf,wooly hat,winter,...,1.287123,1.327414,1.332750,1.316757,0.875426,0.862669,0.667472,0.644763,0.793515,0.605152


In [101]:
# 벡터 컬럼들 드롭
drop_columns = tbl_data.columns[91:181]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'distance_with_words.csv', index=None)

In [102]:
# 단어 컬럼들 드롭
drop_columns = tbl_data.columns[1:91]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

# 단어 없이 coherence만 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'distance.csv', index=None)